In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

In [2]:
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

In [3]:
RAW_DATA_PATH = Path(r"C:\Rainbow\data\raw")
PROCESSED_DATA_PATH = Path(r"C:\Rainbow\data\processed")

In [4]:
# Find all CSV files in raw directory
csv_files = sorted(list(RAW_DATA_PATH.glob("*.csv")))
print(f"\n✓ Found {len(csv_files)} CSV files")

# Display file names and sizes
print("\nFiles discovered:")
for i, file in enumerate(csv_files, 1):
    file_size = file.stat().st_size / (1024 * 1024)  # Convert to MB
    print(f"{i:2d}. {file.name:30s} - {file_size:6.2f} MB")

if len(csv_files) == 0:
    print("\n⚠️  No CSV files found in the raw folder!")
    print("Please ensure your CSV files are in the ../data/raw/ directory")
    raise FileNotFoundError("No CSV files found")



✓ Found 34 CSV files

Files discovered:
 1. Agartala.csv                   -   2.86 MB
 2. Aizwal.csv                     -   2.81 MB
 3. Amravati.csv                   -   2.78 MB
 4. Bengaluru.csv                  -   2.82 MB
 5. Bhopal.csv                     -   2.81 MB
 6. Bhubaneswar.csv                -   2.85 MB
 7. Chandigarh.csv                 -   2.81 MB
 8. Chennai.csv                    -   2.86 MB
 9. Daman.csv                      -   2.83 MB
10. Dehradun.csv                   -   2.81 MB
11. Dispur.csv                     -   2.83 MB
12. Gandhinagar.csv                -   2.84 MB
13. Gangtok.csv                    -   2.82 MB
14. Hyderabad.csv                  -   2.82 MB
15. Imphal.csv                     -   2.82 MB
16. Itanagar.csv                   -   2.82 MB
17. Jaipur.csv                     -   2.81 MB
18. Kavaratti.csv                  -   2.88 MB
19. Kohima.csv                     -   2.82 MB
20. Kolkata.csv                    -   2.86 MB
21. Leh.csv        

In [5]:
sample_file = csv_files[0]
print(f"\nSample file: {sample_file.name}")

df_sample = pd.read_csv(sample_file)

print(f"\n📊 Basic Information:")
print(f"   Rows: {len(df_sample):,}")
print(f"   Columns: {len(df_sample.columns)}")
print(f"\n   Column Names:")
for col in df_sample.columns:
    print(f"      - {col}")

print(f"\n   Data Types:")
print(df_sample.dtypes)

print(f"\n   First 5 Rows:")
print(df_sample.head())

print(f"\n   Last 5 Rows:")
print(df_sample.tail())

print(f"\n   Statistical Summary:")
print(df_sample.describe())

# Check for missing values
print(f"\n   Missing Values:")
missing = df_sample.isnull().sum()
if missing.sum() > 0:
    print(missing[missing > 0])
else:
    print("   ✓ No missing values found in sample file!")


Sample file: Agartala.csv

📊 Basic Information:
   Rows: 43,848
   Columns: 14

   Column Names:
      - YEAR
      - MO
      - DY
      - HR
      - ALLSKY_SFC_SW_DWN
      - CLRSKY_SFC_SW_DWN
      - ALLSKY_KT
      - SZA
      - PRECTOTCORR
      - RH2M
      - T2M
      - WS2M
      - WD2M
      - PS

   Data Types:
YEAR                   int64
MO                     int64
DY                     int64
HR                     int64
ALLSKY_SFC_SW_DWN    float64
CLRSKY_SFC_SW_DWN    float64
ALLSKY_KT            float64
SZA                  float64
PRECTOTCORR          float64
RH2M                 float64
T2M                  float64
WS2M                 float64
WD2M                 float64
PS                   float64
dtype: object

   First 5 Rows:
   YEAR  MO  DY  HR  ALLSKY_SFC_SW_DWN  CLRSKY_SFC_SW_DWN  ALLSKY_KT   SZA  \
0  2020   1   1   0                0.0                0.0     -999.0  90.0   
1  2020   1   1   1                0.0                0.0     -999.0  90.0   
2  2

In [6]:
all_data = []
file_stats = []

for i, file in enumerate(csv_files, 1):
    try:
        # Extract city name from filename (remove .csv extension)
        city_name = file.stem
        
        # Load the CSV
        df = pd.read_csv(file)
        
        # Add metadata columns
        df['city'] = city_name
        df['file_source'] = file.name
        
        # Collect stats
        file_stats.append({
            'city': city_name,
            'rows': len(df),
            'columns': len(df.columns),
            'file_size_mb': file.stat().st_size / (1024 * 1024),
            'missing_values': df.isnull().sum().sum(),
            'date_range_start': df.iloc[0, 0] if len(df) > 0 else 'N/A',
            'date_range_end': df.iloc[-1, 0] if len(df) > 0 else 'N/A'
        })
        
        all_data.append(df)
        print(f"   ✓ [{i:2d}/{len(csv_files)}] Loaded {city_name:25s} - {len(df):,} rows")
        
    except Exception as e:
        print(f"   ✗ Error loading {file.name}: {str(e)}")

# Combine all dataframes
print("\n📦 Combining all datasets...")
df_combined = pd.concat(all_data, ignore_index=True)
print(f"✓ Combined dataset created with {len(df_combined):,} total rows")

# Create file stats dataframe
df_file_stats = pd.DataFrame(file_stats)


   ✓ [ 1/34] Loaded Agartala                  - 43,848 rows
   ✓ [ 2/34] Loaded Aizwal                    - 43,848 rows
   ✓ [ 3/34] Loaded Amravati                  - 43,848 rows
   ✓ [ 4/34] Loaded Bengaluru                 - 43,848 rows
   ✓ [ 5/34] Loaded Bhopal                    - 43,848 rows
   ✓ [ 6/34] Loaded Bhubaneswar               - 43,848 rows
   ✓ [ 7/34] Loaded Chandigarh                - 43,848 rows
   ✓ [ 8/34] Loaded Chennai                   - 43,848 rows
   ✓ [ 9/34] Loaded Daman                     - 43,848 rows
   ✓ [10/34] Loaded Dehradun                  - 43,848 rows
   ✓ [11/34] Loaded Dispur                    - 43,848 rows
   ✓ [12/34] Loaded Gandhinagar               - 43,848 rows
   ✓ [13/34] Loaded Gangtok                   - 43,848 rows
   ✓ [14/34] Loaded Hyderabad                 - 43,848 rows
   ✓ [15/34] Loaded Imphal                    - 43,848 rows
   ✓ [16/34] Loaded Itanagar                  - 43,848 rows
   ✓ [17/34] Loaded Jaipur              

In [7]:
print("\n" + "=" * 80)
print("🔍 DATA QUALITY ASSESSMENT")
print("=" * 80)

print("\n1. File Statistics:")
print(df_file_stats.to_string(index=False))

print(f"\n2. Combined Dataset Overview:")
print(f"   Total Records: {len(df_combined):,}")
print(f"   Total Columns: {len(df_combined.columns)}")
print(f"   Cities Covered: {df_combined['city'].nunique()}")
print(f"   City List: {sorted(df_combined['city'].unique())}")

# Check if there's a date/time column
date_cols = [col for col in df_combined.columns if any(word in col.lower() for word in ['date', 'time', 'timestamp'])]
if date_cols:
    print(f"\n   Date/Time columns found: {date_cols}")
    for col in date_cols:
        try:
            print(f"   Range for '{col}': {df_combined[col].min()} to {df_combined[col].max()}")
        except:
            pass

print(f"\n3. Missing Values Analysis:")
missing_summary = df_combined.isnull().sum()
missing_pct = (missing_summary / len(df_combined) * 100).round(2)
missing_df = pd.DataFrame({
    'Column': missing_summary.index,
    'Missing_Count': missing_summary.values,
    'Missing_Percentage': missing_pct.values
})
missing_df = missing_df[missing_df['Missing_Count'] > 0].sort_values('Missing_Count', ascending=False)

if len(missing_df) > 0:
    print(missing_df.to_string(index=False))
else:
    print("   ✓ No missing values found in the combined dataset!")

print(f"\n4. Data Types:")
print(df_combined.dtypes)

print(f"\n5. Memory Usage:")
print(f"   Total: {df_combined.memory_usage(deep=True).sum() / (1024 * 1024):.2f} MB")



🔍 DATA QUALITY ASSESSMENT

1. File Statistics:
              city  rows  columns  file_size_mb  missing_values  date_range_start  date_range_end
          Agartala 43848       16      2.861824               0              2020            2024
            Aizwal 43848       16      2.809646               0              2020            2024
          Amravati 43848       16      2.781039               0              2020            2024
         Bengaluru 43848       16      2.819387               0              2020            2024
            Bhopal 43848       16      2.811368               0              2020            2024
       Bhubaneswar 43848       16      2.845584               0              2020            2024
        Chandigarh 43848       16      2.813320               0              2020            2024
           Chennai 43848       16      2.856348               0              2020            2024
             Daman 43848       16      2.825051               0       

In [8]:
# Select only numeric columns for analysis
numeric_cols = df_combined.select_dtypes(include=[np.number]).columns.tolist()

if len(numeric_cols) > 0:
    print("\nNumerical Features Summary:")
    print(df_combined[numeric_cols].describe().round(2).T)
    
    # Check for potential outliers
    print("\n6. Outlier Detection (values beyond 3 std deviations):")
    for col in numeric_cols:
        mean = df_combined[col].mean()
        std = df_combined[col].std()
        outliers = df_combined[(df_combined[col] < mean - 3*std) | (df_combined[col] > mean + 3*std)]
        if len(outliers) > 0:
            print(f"   - {col}: {len(outliers)} potential outliers ({len(outliers)/len(df_combined)*100:.2f}%)")
else:
    print("\n⚠️  No numerical columns found for analysis")



Numerical Features Summary:
                       count     mean     std      min      25%      50%  \
YEAR               1490832.0  2022.00    1.41  2020.00  2021.00  2022.00   
MO                 1490832.0     6.52    3.45     1.00     4.00     7.00   
DY                 1490832.0    15.74    8.80     1.00     8.00    16.00   
HR                 1490832.0    11.50    6.92     0.00     5.75    11.50   
ALLSKY_SFC_SW_DWN  1490832.0   192.96  272.40     0.00     0.00     3.03   
CLRSKY_SFC_SW_DWN  1490832.0   245.52  327.18     0.00     0.00     3.58   
ALLSKY_KT          1490832.0  -481.32  499.41  -999.00  -999.00     0.13   
SZA                1490832.0    71.14   24.35     4.05    52.72    88.34   
PRECTOTCORR        1490832.0     4.58   14.54     0.00     0.00     0.04   
RH2M               1490832.0    69.10   23.22     2.40    53.85    74.58   
T2M                1490832.0    22.93    9.26   -39.37    18.73    25.25   
WS2M               1490832.0     2.03    1.53     0.00     

In [9]:
# Create results/figures directory if it doesn't exist
figures_path = Path("/content/RAINBOW/results/figures")
figures_path.mkdir(parents=True, exist_ok=True)

# Figure 1: Data Quality Overview
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# Plot 1: Records per city
city_counts = df_combined['city'].value_counts().sort_values(ascending=True)
axes[0, 0].barh(range(len(city_counts)), city_counts.values, color='skyblue')
axes[0, 0].set_yticks(range(len(city_counts)))
axes[0, 0].set_yticklabels(city_counts.index, fontsize=8)
axes[0, 0].set_xlabel('Number of Records', fontsize=10)
axes[0, 0].set_title('Records per State Capital', fontsize=12, fontweight='bold')
axes[0, 0].grid(axis='x', alpha=0.3)

# Plot 2: Missing values heatmap (if any)
if len(missing_df) > 0:
    missing_cols = missing_df['Column'].tolist()[:20]  # Top 20 columns with missing values
    missing_matrix = df_combined[missing_cols].isnull().sample(min(1000, len(df_combined))).T
    sns.heatmap(missing_matrix, cbar=True, cmap='YlOrRd', ax=axes[0, 1], 
                xticklabels=False, yticklabels=True)
    axes[0, 1].set_title('Missing Values Pattern (Sample)', fontsize=12, fontweight='bold')
    axes[0, 1].set_ylabel('Columns', fontsize=10)
else:
    axes[0, 1].text(0.5, 0.5, 'No Missing Values ✓', 
                     ha='center', va='center', fontsize=16, transform=axes[0, 1].transAxes,
                     color='green', weight='bold')
    axes[0, 1].set_title('Missing Values Pattern', fontsize=12, fontweight='bold')
    axes[0, 1].axis('off')

# Plot 3: File size distribution
axes[1, 0].bar(range(len(df_file_stats)), df_file_stats['file_size_mb'], color='lightcoral')
axes[1, 0].set_xticks(range(len(df_file_stats)))
axes[1, 0].set_xticklabels(df_file_stats['city'], rotation=90, fontsize=7)
axes[1, 0].set_ylabel('File Size (MB)', fontsize=10)
axes[1, 0].set_title('File Size Distribution', fontsize=12, fontweight='bold')
axes[1, 0].grid(axis='y', alpha=0.3)

# Plot 4: Data completeness score
completeness = (1 - df_file_stats['missing_values'] / (df_file_stats['rows'] * df_file_stats['columns'])) * 100
axes[1, 1].bar(range(len(df_file_stats)), completeness, color='lightgreen')
axes[1, 1].set_xticks(range(len(df_file_stats)))
axes[1, 1].set_xticklabels(df_file_stats['city'], rotation=90, fontsize=7)
axes[1, 1].set_ylabel('Completeness (%)', fontsize=10)
axes[1, 1].set_title('Data Completeness by City', fontsize=12, fontweight='bold')
axes[1, 1].axhline(y=95, color='r', linestyle='--', label='95% threshold', linewidth=2)
axes[1, 1].legend()
axes[1, 1].grid(axis='y', alpha=0.3)

plt.tight_layout()
plt.savefig(figures_path / 'data_quality_overview.png', dpi=300, bbox_inches='tight')
print("✓ Saved: data_quality_overview.png")

# Figure 2: Distribution of key meteorological parameters
if len(numeric_cols) >= 4:
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    
    # Select first 4 numeric columns for visualization
    for idx, col in enumerate(numeric_cols[:4]):
        row = idx // 2
        col_idx = idx % 2
        
        data_to_plot = df_combined[col].dropna()
        axes[row, col_idx].hist(data_to_plot, bins=50, 
                                 color='steelblue', edgecolor='black', alpha=0.7)
        axes[row, col_idx].set_xlabel(col, fontsize=10)
        axes[row, col_idx].set_ylabel('Frequency', fontsize=10)
        axes[row, col_idx].set_title(f'Distribution of {col}', fontsize=12, fontweight='bold')
        axes[row, col_idx].grid(axis='y', alpha=0.3)
        
        # Add statistics text
        stats_text = f'Mean: {data_to_plot.mean():.2f}\nStd: {data_to_plot.std():.2f}'
        axes[row, col_idx].text(0.98, 0.98, stats_text, 
                                transform=axes[row, col_idx].transAxes,
                                verticalalignment='top', horizontalalignment='right',
                                bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5),
                                fontsize=9)
    
    plt.tight_layout()
    plt.savefig(figures_path / 'parameter_distributions.png', dpi=300, bbox_inches='tight')
    print("✓ Saved: parameter_distributions.png")

plt.close('all')

✓ Saved: data_quality_overview.png
✓ Saved: parameter_distributions.png


In [11]:
output_file = PROCESSED_DATA_PATH / "combined_nasa_power_data.csv"
df_combined.to_csv(output_file, index=False)
print(f"✓ Saved consolidated dataset: {output_file}")
print(f"  Size: {output_file.stat().st_size / (1024 * 1024):.2f} MB")
print(f"  Records: {len(df_combined):,}")
print(f"  Columns: {len(df_combined.columns)}")

# Save file statistics
stats_file = PROCESSED_DATA_PATH / "data_collection_stats.csv"
df_file_stats.to_csv(stats_file, index=False)
print(f"✓ Saved file statistics: {stats_file}")

# Save column information
columns_info = pd.DataFrame({
    'Column_Name': df_combined.columns,
    'Data_Type': df_combined.dtypes.values,
    'Non_Null_Count': df_combined.count().values,
    'Null_Count': df_combined.isnull().sum().values,
    'Null_Percentage': (df_combined.isnull().sum() / len(df_combined) * 100).round(2).values
})
columns_file = PROCESSED_DATA_PATH / "columns_info.csv"
columns_info.to_csv(columns_file, index=False)
print(f"✓ Saved column information: {columns_file}")



✓ Saved consolidated dataset: C:\Rainbow\data\processed\combined_nasa_power_data.csv
  Size: 127.96 MB
  Records: 1,490,832
  Columns: 16
✓ Saved file statistics: C:\Rainbow\data\processed\data_collection_stats.csv
✓ Saved column information: C:\Rainbow\data\processed\columns_info.csv


In [14]:
print("\n" + "=" * 80)
print("SUMMARY REPORT")
print("=" * 80)

summary_report = f"""
DATA COLLECTION SUMMARY
{'=' * 80}

1. DATA COLLECTION
   - Files Loaded: {len(csv_files)}
   - State Capitals Covered: {df_combined['city'].nunique()}
   - Total Records: {len(df_combined):,}
   - Total Features: {len(df_combined.columns) - 2}  (excluding city and file_source)
   
2. DATA QUALITY
   - Overall Completeness: {((1 - df_combined.isnull().sum().sum() / (len(df_combined) * len(df_combined.columns))) * 100):.2f}%
   - Files with Missing Data: {len(df_file_stats[df_file_stats['missing_values'] > 0])}
   - Average Records per City: {len(df_combined) / df_combined['city'].nunique():.0f}
   
3. CITIES INCLUDED
   {', '.join(sorted(df_combined['city'].unique()))}
   
4. NEXT STEPS
   [DONE] Data successfully consolidated
   [TODO] Need to add rainbow observation labels (Target variable)
   [TODO] Perform feature engineering (solar position, time features)
   [TODO] Data cleaning and preprocessing
   [TODO] Exploratory Data Analysis (EDA)
   
5. FILES CREATED
   - Combined dataset: {output_file}
   - File statistics: {stats_file}
   - Column information: {columns_file}
   - Visualizations: {figures_path}/
   
{'=' * 80}
"""

print(summary_report)

# Save summary report
report_file = PROCESSED_DATA_PATH / "data_exploration_summary.txt"
with open(report_file, 'w', encoding='utf-8') as f:
    f.write(summary_report)
print(f"[DONE] Saved summary report: {report_file}")

print("\n" + "=" * 80)
print("DATA EXPLORATION COMPLETE!")
print("=" * 80)
print("\nYou can now proceed to:")
print("1. Review the visualizations in ../../results/figures/")
print("2. Check the combined dataset in ../../data/processed/")
print("3. Move to the next step: Feature Engineering or Rainbow Labeling")
print("=" * 80)

# Display final dataframe info
print("\n" + "=" * 80)
print("FINAL COMBINED DATASET INFO")
print("=" * 80)
print(df_combined.info())


SUMMARY REPORT

DATA COLLECTION SUMMARY

1. DATA COLLECTION
   - Files Loaded: 34
   - State Capitals Covered: 34
   - Total Records: 1,490,832
   - Total Features: 14  (excluding city and file_source)

2. DATA QUALITY
   - Overall Completeness: 100.00%
   - Files with Missing Data: 0
   - Average Records per City: 43848

3. CITIES INCLUDED
   Agartala, Aizwal, Amravati, Bengaluru, Bhopal, Bhubaneswar, Chandigarh, Chennai, Daman, Dehradun, Dispur, Gandhinagar, Gangtok, Hyderabad, Imphal, Itanagar, Jaipur, Kavaratti, Kohima, Kolkata, Leh, Lucknow, Mumbai, New_Delhi, Panaji, Patna, Port_Blair, Puducherry, Raipur, Ranchi, Shillong, Shimla, Srinagar, Thiruvananthapuram

4. NEXT STEPS
   [DONE] Data successfully consolidated
   [TODO] Need to add rainbow observation labels (Target variable)
   [TODO] Perform feature engineering (solar position, time features)
   [TODO] Data cleaning and preprocessing
   [TODO] Exploratory Data Analysis (EDA)

5. FILES CREATED
   - Combined dataset: C:\Rain